# DL Assignment 2: ARC-AGI Open Architecture Design Challenge
**Team: Gradient Ascend**

## Architecture: Iterative Refinement Transformer
- **Context Encoder**: Bidirectional transformer over demo pairs + test input
- **Iterative Decoder**: Cross-attending decoder with shared weights, run T times
- **Positional Encoding**: 3D RoPE (row, col, pair_index)
- **Loss**: Deep-supervised cross-entropy at every refinement step
- **Inference**: Greedy (attempt 1) + Test-Time Training (attempt 2)
- **Parameters**: ~36.8M (budget: ≤50M)

## Section 1: Setup & Data Loading

In [ ]:
# Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/arc_agi/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

In [ ]:
# Clone ARC-AGI dataset
!git clone --depth 1 https://github.com/fchollet/ARC-AGI.git /content/ARC-AGI 2>/dev/null || echo 'Already cloned'
TRAIN_DIR = '/content/ARC-AGI/data/training'
EVAL_DIR = '/content/ARC-AGI/data/evaluation'
print(f'Training tasks: {len(os.listdir(TRAIN_DIR))}')
print(f'Evaluation tasks: {len(os.listdir(EVAL_DIR))}')

In [ ]:
"""Data Loading, Tokenization, and Augmentation."""

import json
import os
import random
import copy
import math
import time
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
MAX_GRID_DIM = 30
NUM_COLORS = 10
MAX_SEQ_LEN = 512

PAD_TOKEN = 10
ROW_SEP = 11
GRID_SEP = 12
PAIR_SEP = 13
VOCAB_SIZE = 14  # 0-9 colors + 4 special tokens


# ---------------------------------------------------------------------------
# Task Loading
# ---------------------------------------------------------------------------
class ARCTask:
    """Represents a single ARC-AGI task with demo pairs and test pair(s)."""

    def __init__(self, task_id: str, train_pairs: List[Dict], test_pairs: List[Dict]):
        self.task_id = task_id
        self.train_pairs = train_pairs
        self.test_pairs = test_pairs

    @classmethod
    def from_json(cls, filepath: str) -> 'ARCTask':
        with open(filepath, 'r') as f:
            data = json.load(f)
        task_id = Path(filepath).stem
        return cls(task_id=task_id, train_pairs=data['train'], test_pairs=data['test'])

    def __repr__(self):
        return f'ARCTask(id={self.task_id}, demos={len(self.train_pairs)}, tests={len(self.test_pairs)})'


def load_tasks(directory: str) -> List[ARCTask]:
    """Load all ARC tasks from a directory of JSON files."""
    tasks = []
    for fname in sorted(os.listdir(directory)):
        if fname.endswith('.json'):
            tasks.append(ARCTask.from_json(os.path.join(directory, fname)))
    return tasks


# ---------------------------------------------------------------------------
# Output Shape Prediction
# ---------------------------------------------------------------------------
def predict_output_shape(task: ARCTask, test_idx: int = 0) -> Tuple[int, int]:
    """
    Heuristically predict test output grid shape.
    Priority: same-as-input > uniform-output-shape > test-input-shape.
    """
    demos = task.train_pairs
    test_input = task.test_pairs[test_idx]['input']
    same_as_input = all(
        len(d['output']) == len(d['input']) and len(d['output'][0]) == len(d['input'][0])
        for d in demos
    )
    if same_as_input:
        return len(test_input), len(test_input[0])
    out_shapes = [(len(d['output']), len(d['output'][0])) for d in demos]
    if len(set(out_shapes)) == 1:
        return out_shapes[0]
    return len(test_input), len(test_input[0])


# ---------------------------------------------------------------------------
# Grid Tokenization
# ---------------------------------------------------------------------------
def grid_to_tokens(grid: List[List[int]]) -> Tuple[List[int], List[Tuple[int, int]]]:
    """Flatten a 2D grid into 1D tokens with (row, col) positions."""
    tokens, positions = [], []
    for r, row in enumerate(grid):
        if r > 0:
            tokens.append(ROW_SEP)
            positions.append((r, -1))
        for c, val in enumerate(row):
            tokens.append(val)
            positions.append((r, c))
    return tokens, positions


def build_task_sequence(task: ARCTask, test_idx: int = 0, max_len: int = MAX_SEQ_LEN) -> Dict[str, torch.Tensor]:
    """
    Build the full input sequence for a task.
    Layout: [PAIR_SEP][in1][GRID_SEP][out1][PAIR_SEP][in2][GRID_SEP][out2]...[PAIR_SEP][test_in][GRID_SEP]
    """
    ctx_tokens, ctx_rows, ctx_cols, ctx_pairs = [], [], [], []

    # Encode demo pairs
    for pair_idx, pair in enumerate(task.train_pairs):
        ctx_tokens.append(PAIR_SEP); ctx_rows.append(0); ctx_cols.append(0); ctx_pairs.append(pair_idx)
        itok, ipos = grid_to_tokens(pair['input'])
        ctx_tokens.extend(itok); ctx_rows.extend([p[0] for p in ipos]); ctx_cols.extend([p[1] for p in ipos]); ctx_pairs.extend([pair_idx]*len(itok))
        ctx_tokens.append(GRID_SEP); ctx_rows.append(0); ctx_cols.append(0); ctx_pairs.append(pair_idx)
        otok, opos = grid_to_tokens(pair['output'])
        ctx_tokens.extend(otok); ctx_rows.extend([p[0] for p in opos]); ctx_cols.extend([p[1] for p in opos]); ctx_pairs.extend([pair_idx]*len(otok))

    # Test input
    test_pair_idx = len(task.train_pairs)
    ctx_tokens.append(PAIR_SEP); ctx_rows.append(0); ctx_cols.append(0); ctx_pairs.append(test_pair_idx)
    test_input = task.test_pairs[test_idx]['input']
    titok, tipos = grid_to_tokens(test_input)
    ctx_tokens.extend(titok); ctx_rows.extend([p[0] for p in tipos]); ctx_cols.extend([p[1] for p in tipos]); ctx_pairs.extend([test_pair_idx]*len(titok))
    ctx_tokens.append(GRID_SEP); ctx_rows.append(0); ctx_cols.append(0); ctx_pairs.append(test_pair_idx)

    # Truncate
    ctx_tokens = ctx_tokens[:max_len]; ctx_rows = ctx_rows[:max_len]; ctx_cols = ctx_cols[:max_len]; ctx_pairs = ctx_pairs[:max_len]

    # Target output
    test_output = task.test_pairs[test_idx]['output']
    out_h, out_w = predict_output_shape(task, test_idx)
    target_tokens, target_rows, target_cols = [], [], []
    for r in range(out_h):
        for c in range(out_w):
            target_tokens.append(test_output[r][c] if r < len(test_output) and c < len(test_output[r]) else 0)
            target_rows.append(r); target_cols.append(c)

    # Test input flat
    ti_flat_tokens, ti_flat_rows, ti_flat_cols = [], [], []
    for r, row in enumerate(test_input):
        for c, val in enumerate(row):
            ti_flat_tokens.append(val); ti_flat_rows.append(r); ti_flat_cols.append(c)

    return {
        'context_tokens': torch.tensor(ctx_tokens, dtype=torch.long),
        'context_rows': torch.tensor(ctx_rows, dtype=torch.long),
        'context_cols': torch.tensor(ctx_cols, dtype=torch.long),
        'context_pairs': torch.tensor(ctx_pairs, dtype=torch.long),
        'target_tokens': torch.tensor(target_tokens, dtype=torch.long),
        'target_rows': torch.tensor(target_rows, dtype=torch.long),
        'target_cols': torch.tensor(target_cols, dtype=torch.long),
        'test_input_tokens': torch.tensor(ti_flat_tokens, dtype=torch.long),
        'test_input_rows': torch.tensor(ti_flat_rows, dtype=torch.long),
        'test_input_cols': torch.tensor(ti_flat_cols, dtype=torch.long),
        'output_h': out_h, 'output_w': out_w,
    }

print('Data loading code defined.')

In [ ]:
"""Data Augmentation: Dihedral group transforms + color permutation."""

def rotate_grid_90(grid):
    """Rotate grid 90 degrees clockwise."""
    h, w = len(grid), len(grid[0])
    return [[grid[h - 1 - c][r] for c in range(h)] for r in range(w)]

def flip_grid_horizontal(grid):
    """Flip grid left-right."""
    return [row[::-1] for row in grid]

def apply_dihedral(grid, transform_id):
    """Apply one of 8 D4 transforms: 0=identity, 1-3=rotations, 4=flip, 5-7=flip+rotations."""
    g = [row[:] for row in grid]
    if transform_id >= 4:
        g = flip_grid_horizontal(g)
    for _ in range(transform_id % 4):
        g = rotate_grid_90(g)
    return g

def permute_colors(grid, perm):
    """Apply color permutation. Color 0 (black) stays fixed."""
    return [[perm.get(val, val) for val in row] for row in grid]

def random_color_permutation():
    """Random permutation of non-zero colors 1-9."""
    colors = list(range(1, 10))
    shuffled = colors[:]
    random.shuffle(shuffled)
    return {orig: new for orig, new in zip(colors, shuffled)}

def augment_task(task, dihedral_id=0, color_perm=None):
    """Return augmented ARCTask with transforms applied consistently to all grids."""
    def transform(grid):
        g = apply_dihedral(grid, dihedral_id)
        if color_perm is not None:
            g = permute_colors(g, color_perm)
        return g
    new_train = [{'input': transform(p['input']), 'output': transform(p['output'])} for p in task.train_pairs]
    new_test = [{'input': transform(p['input']), 'output': transform(p['output'])} for p in task.test_pairs]
    return ARCTask(task_id=task.task_id, train_pairs=new_train, test_pairs=new_test)


class ARCDataset(Dataset):
    """ARC dataset with on-the-fly augmentation."""
    def __init__(self, tasks, augment=True, n_dihedral=8, n_color_perms=2, max_seq_len=MAX_SEQ_LEN):
        self.tasks = tasks
        self.augment = augment
        self.n_dihedral = n_dihedral if augment else 1
        self.n_color_perms = n_color_perms if augment else 1
        self.max_seq_len = max_seq_len
        self.augments_per_task = self.n_dihedral * self.n_color_perms

    def __len__(self):
        return len(self.tasks) * self.augments_per_task

    def __getitem__(self, idx):
        task_idx = idx // self.augments_per_task
        aug_idx = idx % self.augments_per_task
        dihedral_id = aug_idx // self.n_color_perms
        color_idx = aug_idx % self.n_color_perms
        task = self.tasks[task_idx]
        if self.augment and (dihedral_id > 0 or color_idx > 0):
            color_perm = random_color_permutation() if color_idx > 0 else None
            task = augment_task(task, dihedral_id=dihedral_id, color_perm=color_perm)
        seq = build_task_sequence(task, test_idx=0, max_len=self.max_seq_len)
        seq['task_idx'] = task_idx
        return seq


def collate_arc_batch(batch):
    """Collate variable-length task sequences into padded batch."""
    max_ctx = max(b['context_tokens'].size(0) for b in batch)
    max_tgt = max(b['target_tokens'].size(0) for b in batch)
    max_ti = max(b['test_input_tokens'].size(0) for b in batch)
    B = len(batch)
    result = {
        'context_tokens': torch.full((B, max_ctx), PAD_TOKEN, dtype=torch.long),
        'context_rows': torch.zeros(B, max_ctx, dtype=torch.long),
        'context_cols': torch.zeros(B, max_ctx, dtype=torch.long),
        'context_pairs': torch.zeros(B, max_ctx, dtype=torch.long),
        'context_mask': torch.zeros(B, max_ctx, dtype=torch.bool),
        'target_tokens': torch.zeros(B, max_tgt, dtype=torch.long),
        'target_rows': torch.zeros(B, max_tgt, dtype=torch.long),
        'target_cols': torch.zeros(B, max_tgt, dtype=torch.long),
        'target_mask': torch.zeros(B, max_tgt, dtype=torch.bool),
        'test_input_tokens': torch.full((B, max_ti), PAD_TOKEN, dtype=torch.long),
        'test_input_rows': torch.zeros(B, max_ti, dtype=torch.long),
        'test_input_cols': torch.zeros(B, max_ti, dtype=torch.long),
        'test_input_mask': torch.zeros(B, max_ti, dtype=torch.bool),
        'output_h': torch.zeros(B, dtype=torch.long),
        'output_w': torch.zeros(B, dtype=torch.long),
        'task_idx': torch.zeros(B, dtype=torch.long),
    }
    for i, b in enumerate(batch):
        lc = b['context_tokens'].size(0)
        result['context_tokens'][i, :lc] = b['context_tokens']
        result['context_rows'][i, :lc] = b['context_rows']
        result['context_cols'][i, :lc] = b['context_cols']
        result['context_pairs'][i, :lc] = b['context_pairs']
        result['context_mask'][i, :lc] = True
        lt = b['target_tokens'].size(0)
        result['target_tokens'][i, :lt] = b['target_tokens']
        result['target_rows'][i, :lt] = b['target_rows']
        result['target_cols'][i, :lt] = b['target_cols']
        result['target_mask'][i, :lt] = True
        li = b['test_input_tokens'].size(0)
        result['test_input_tokens'][i, :li] = b['test_input_tokens']
        result['test_input_rows'][i, :li] = b['test_input_rows']
        result['test_input_cols'][i, :li] = b['test_input_cols']
        result['test_input_mask'][i, :li] = True
        result['output_h'][i] = b['output_h']
        result['output_w'][i] = b['output_w']
        result['task_idx'][i] = b['task_idx']
    return result

print('Augmentation & dataset code defined.')

In [ ]:
# Load and inspect training data
all_tasks = load_tasks(TRAIN_DIR)
print(f'Loaded {len(all_tasks)} training tasks')

# Train/val split: 360 train, 40 validation
random.seed(42)
indices = list(range(len(all_tasks)))
random.shuffle(indices)
val_indices = set(indices[:40])
train_tasks = [all_tasks[i] for i in range(len(all_tasks)) if i not in val_indices]
val_tasks = [all_tasks[i] for i in val_indices]
print(f'Train: {len(train_tasks)}, Val: {len(val_tasks)}')

# Sequence length statistics
lengths = [build_task_sequence(t)['context_tokens'].shape[0] for t in all_tasks]
print(f'Context seq lengths: min={min(lengths)}, median={sorted(lengths)[200]}, max={max(lengths)}')
print(f'Tasks fitting in 512 tokens: {sum(1 for l in lengths if l < 512)}/400 ({sum(1 for l in lengths if l < 512)/4:.1f}%)')

## Section 2: Model Architecture

In [ ]:
"""3D Rotary Positional Encoding (row, col, pair_index)."""

class RoPE3D(nn.Module):
    """
    3-axis Rotary Positional Encoding for (row, col, pair_index).
    Splits d_model into 3 chunks, applies standard 1D RoPE per axis.
    Encodes relative spatial distances naturally — critical for ARC
    where rules are fundamentally spatial.
    """
    def __init__(self, d_model, max_positions=64, base=10000.0):
        super().__init__()
        self.d_model = d_model
        d_per_axis = d_model // 3
        self.d_row = d_model - 2 * d_per_axis
        self.d_col = d_per_axis
        self.d_pair = d_per_axis
        self.register_buffer('inv_freq_row', self._make_inv_freq(self.d_row, base))
        self.register_buffer('inv_freq_col', self._make_inv_freq(self.d_col, base))
        self.register_buffer('inv_freq_pair', self._make_inv_freq(self.d_pair, base))

    @staticmethod
    def _make_inv_freq(dim, base):
        half = dim // 2
        if half == 0:
            return torch.zeros(1)
        return 1.0 / (base ** (torch.arange(0, half, dtype=torch.float32) * 2.0 / dim))

    def _apply_rope_1d(self, x, positions, inv_freq):
        D = x.shape[-1]
        half = D // 2
        if half == 0:
            return x
        freqs = positions.unsqueeze(-1).float() * inv_freq
        cos_f = freqs.cos()
        sin_f = freqs.sin()
        x1 = x[..., :half]
        x2 = x[..., half:2*half]
        rotated = torch.cat([x1*cos_f - x2*sin_f, x2*cos_f + x1*sin_f], dim=-1)
        if D > 2 * half:
            rotated = torch.cat([rotated, x[..., 2*half:]], dim=-1)
        return rotated

    def forward(self, x, rows, cols, pairs):
        d1 = self.d_row
        d2 = d1 + self.d_col
        x_row = self._apply_rope_1d(x[..., :d1], rows, self.inv_freq_row)
        x_col = self._apply_rope_1d(x[..., d1:d2], cols, self.inv_freq_col)
        x_pair = self._apply_rope_1d(x[..., d2:], pairs, self.inv_freq_pair)
        return torch.cat([x_row, x_col, x_pair], dim=-1)


def apply_rope_to_qk(rope, q, k, q_rows, q_cols, q_pairs, k_rows, k_cols, k_pairs):
    """Apply RoPE per-head to Q and K tensors. q,k: (B, H, L, d_head)."""
    B, H, L_q, D = q.shape
    _, _, L_k, _ = k.shape
    q_flat = q.reshape(B * H, L_q, D)
    k_flat = k.reshape(B * H, L_k, D)
    q_rows_exp = q_rows.unsqueeze(1).expand(-1, H, -1).reshape(B*H, L_q)
    q_cols_exp = q_cols.unsqueeze(1).expand(-1, H, -1).reshape(B*H, L_q)
    q_pairs_exp = q_pairs.unsqueeze(1).expand(-1, H, -1).reshape(B*H, L_q)
    k_rows_exp = k_rows.unsqueeze(1).expand(-1, H, -1).reshape(B*H, L_k)
    k_cols_exp = k_cols.unsqueeze(1).expand(-1, H, -1).reshape(B*H, L_k)
    k_pairs_exp = k_pairs.unsqueeze(1).expand(-1, H, -1).reshape(B*H, L_k)
    q_rot = rope(q_flat, q_rows_exp, q_cols_exp, q_pairs_exp)
    k_rot = rope(k_flat, k_rows_exp, k_cols_exp, k_pairs_exp)
    return q_rot.reshape(B, H, L_q, D), k_rot.reshape(B, H, L_k, D)

print('RoPE3D defined.')

In [ ]:
"""Transformer Building Blocks: Attention, FFN, Encoder/Decoder blocks."""

class MultiHeadAttention(nn.Module):
    """Multi-head attention with 3D RoPE support and optional cross-attention."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, rope=None, x_rows=None, x_cols=None, x_pairs=None,
                kv=None, kv_rows=None, kv_cols=None, kv_pairs=None, key_padding_mask=None):
        B, L_q, _ = x.shape
        if kv is None:
            kv = x; kv_rows = x_rows; kv_cols = x_cols; kv_pairs = x_pairs
        L_k = kv.shape[1]
        q = self.q_proj(x).view(B, L_q, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(kv).view(B, L_k, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(kv).view(B, L_k, self.n_heads, self.d_head).transpose(1, 2)
        if rope is not None and x_rows is not None:
            q, k = apply_rope_to_qk(rope, q, k, x_rows, x_cols, x_pairs, kv_rows, kv_cols, kv_pairs)
        scale = math.sqrt(self.d_head)
        attn = torch.matmul(q, k.transpose(-2, -1)) / scale
        if key_padding_mask is not None:
            mask = ~key_padding_mask
            attn = attn.masked_fill(mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).reshape(B, L_q, self.d_model)
        return self.out_proj(out)


class FeedForward(nn.Module):
    """SwiGLU-style feed-forward network."""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w_gate(x)) * self.w1(x)))


class EncoderBlock(nn.Module):
    """Pre-norm transformer encoder block."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, rope, rows, cols, pairs, mask=None):
        h = self.norm1(x)
        x = x + self.attn(h, rope=rope, x_rows=rows, x_cols=cols, x_pairs=pairs, key_padding_mask=mask)
        h = self.norm2(x)
        x = x + self.ffn(h)
        return x


class DecoderBlock(nn.Module):
    """Pre-norm decoder block with self-attention + cross-attention."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, rope, x_rows, x_cols, x_pairs,
                encoder_out, enc_rows, enc_cols, enc_pairs, enc_mask=None):
        h = self.norm1(x)
        x = x + self.self_attn(h, rope=rope, x_rows=x_rows, x_cols=x_cols, x_pairs=x_pairs)
        h = self.norm2(x)
        x = x + self.cross_attn(h, rope=rope, x_rows=x_rows, x_cols=x_cols, x_pairs=x_pairs,
                                kv=encoder_out, kv_rows=enc_rows, kv_cols=enc_cols, kv_pairs=enc_pairs,
                                key_padding_mask=enc_mask)
        h = self.norm3(x)
        x = x + self.ffn(h)
        return x

print('Transformer blocks defined.')

In [ ]:
"""Full Model: ContextEncoder + IterativeDecoder + ARCModel wrapper."""

class ContextEncoder(nn.Module):
    """Bidirectional transformer encoder for demo pairs + test input."""
    def __init__(self, d_model=384, n_heads=8, n_layers=8, d_ff=1536, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, d_model, padding_idx=PAD_TOKEN)
        self.rope = RoPE3D(d_model // n_heads)
        self.layers = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, tokens, rows, cols, pairs, mask=None):
        x = self.embedding(tokens)
        for layer in self.layers:
            x = layer(x, self.rope, rows, cols, pairs, mask)
        return self.final_norm(x)


class IterativeDecoder(nn.Module):
    """
    Cross-attending decoder with shared weights, run T times.
    Each step refines the output guess by cross-attending to encoder output.
    """
    def __init__(self, d_model=384, n_heads=8, n_layers=6, d_ff=1536, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.guess_embedding = nn.Embedding(NUM_COLORS, d_model)
        self.input_embedding = nn.Embedding(VOCAB_SIZE, d_model, padding_idx=PAD_TOKEN)
        self.type_embed = nn.Embedding(2, d_model)  # 0=test input, 1=guess
        self.rope = RoPE3D(d_model // n_heads)
        self.layers = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, NUM_COLORS, bias=False)

    def forward_one_step(self, guess_tokens, guess_rows, guess_cols,
                         test_input_tokens, test_input_rows, test_input_cols, test_input_mask,
                         encoder_out, enc_rows, enc_cols, enc_pairs, enc_mask=None):
        """Single decoder pass. Returns (B, L_guess, NUM_COLORS) logits."""
        B = guess_tokens.shape[0]
        L_guess = guess_tokens.shape[1]
        L_in = test_input_tokens.shape[1]
        dev = guess_tokens.device

        inp_emb = self.input_embedding(test_input_tokens) + self.type_embed(torch.zeros(B, L_in, dtype=torch.long, device=dev))
        guess_emb = self.guess_embedding(guess_tokens) + self.type_embed(torch.ones(B, L_guess, dtype=torch.long, device=dev))
        x = torch.cat([inp_emb, guess_emb], dim=1)

        pair_idx_val = 5  # distinct from demo pair indices
        dec_rows = torch.cat([test_input_rows, guess_rows], dim=1)
        dec_cols = torch.cat([test_input_cols, guess_cols], dim=1)
        dec_pairs = torch.full((B, L_in + L_guess), pair_idx_val, dtype=torch.long, device=dev)

        for layer in self.layers:
            x = layer(x, self.rope, dec_rows, dec_cols, dec_pairs,
                      encoder_out, enc_rows, enc_cols, enc_pairs, enc_mask)
        x = self.final_norm(x)
        return self.output_head(x[:, L_in:, :])


class PerTaskEmbedding(nn.Module):
    """Learnable per-task vectors added to encoder input during training."""
    def __init__(self, n_tasks, d_model):
        super().__init__()
        self.embeddings = nn.Embedding(n_tasks, d_model)
        nn.init.normal_(self.embeddings.weight, std=0.02)

    def forward(self, task_indices, seq_len):
        return self.embeddings(task_indices).unsqueeze(1).expand(-1, seq_len, -1)


class ARCModel(nn.Module):
    """
    Iterative Refinement Transformer for ARC-AGI.
    1. ContextEncoder processes demo pairs + test input (once)
    2. IterativeDecoder refines output guess over T steps (shared weights)
    3. Returns logits at every step for deep supervision
    """
    def __init__(self, d_model=384, enc_layers=8, dec_layers=6, n_heads=8,
                 d_ff=1536, n_train_tasks=400, dropout=0.1, refine_steps=5):
        super().__init__()
        self.d_model = d_model
        self.refine_steps = refine_steps
        self.encoder = ContextEncoder(d_model, n_heads, enc_layers, d_ff, dropout)
        self.decoder = IterativeDecoder(d_model, n_heads, dec_layers, d_ff, dropout)
        self.task_embed = PerTaskEmbedding(n_train_tasks, d_model)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, context_tokens, context_rows, context_cols, context_pairs, context_mask,
                target_rows, target_cols, target_mask,
                test_input_tokens, test_input_rows, test_input_cols, test_input_mask,
                task_idx=None, T=None):
        if T is None: T = self.refine_steps
        B = context_tokens.shape[0]
        L_out = target_rows.shape[1]

        # Encode context (once)
        enc_input = self.encoder.embedding(context_tokens)
        if task_idx is not None:
            enc_input = enc_input + self.task_embed(task_idx, enc_input.shape[1])
        x = enc_input
        for layer in self.encoder.layers:
            x = layer(x, self.encoder.rope, context_rows, context_cols, context_pairs, context_mask)
        encoder_out = self.encoder.final_norm(x)

        # Iterative decoding
        guess = torch.zeros(B, L_out, dtype=torch.long, device=context_tokens.device)
        all_logits = []
        for t in range(T):
            logits = self.decoder.forward_one_step(
                guess, target_rows, target_cols,
                test_input_tokens, test_input_rows, test_input_cols, test_input_mask,
                encoder_out, context_rows, context_cols, context_pairs, context_mask)
            all_logits.append(logits)
            guess = logits.argmax(dim=-1).detach()
        return all_logits

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

print('Model architecture defined.')

In [ ]:
# ============================================================
# PARAMETER COUNT (required by assignment)
# ============================================================

def print_parameter_table(model):
    """Pretty-print the parameter budget breakdown."""
    total = model.count_parameters()
    print(f"{'Component':<30} {'Parameters':>15}")
    print('-' * 47)
    for name, child in model.named_children():
        n = sum(p.numel() for p in child.parameters() if p.requires_grad)
        pct = n / total * 100
        print(f'{name:<30} {n:>12,} ({pct:5.1f}%)')
    print('-' * 47)
    print(f'{"TOTAL":<30} {total:>12,}')
    print(f'{"Budget (50M)":<30} {50_000_000:>12,}')
    assert total <= 50_000_000, f'OVER BUDGET! {total:,} > 50,000,000'
    print('\u2713 Within 50M parameter budget')

# Instantiate full model
model = ARCModel(
    d_model=384, enc_layers=8, dec_layers=6, n_heads=8,
    d_ff=1536, n_train_tasks=len(train_tasks), dropout=0.1, refine_steps=5
)
print_parameter_table(model)

## Section 3: Training

In [ ]:
"""Loss function and LR scheduler."""

def deep_supervised_loss(all_logits, target, mask):
    """
    Deep-supervised CE loss across all refinement steps.
    Later steps weighted more: L = sum_t 2^(t-T+1) * CE_t
    This is the PRIMARY performance driver for iterative refinement
    (per TRM/HRM literature).
    """
    T = len(all_logits)
    total_loss = torch.tensor(0.0, device=target.device)
    for t, logits in enumerate(all_logits):
        weight = 2.0 ** (t - T + 1)
        B, L, C = logits.shape
        flat_logits = logits.reshape(-1, C)
        flat_target = target.reshape(-1)
        flat_mask = mask.reshape(-1).float()
        ce = F.cross_entropy(flat_logits, flat_target, reduction='none')
        masked_ce = (ce * flat_mask).sum() / flat_mask.sum().clamp(min=1)
        total_loss = total_loss + weight * masked_ce
    return total_loss


class WSDScheduler:
    """Warmup-Stable-Decay LR scheduler."""
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr_ratio=0.1):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr_ratio = min_lr_ratio
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.step_count = 0

    def step(self):
        self.step_count += 1
        if self.step_count <= self.warmup_steps:
            scale = self.step_count / max(1, self.warmup_steps)
        else:
            progress = (self.step_count - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            scale = max(self.min_lr_ratio, 1.0 - progress * (1.0 - self.min_lr_ratio))
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * scale

    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]

print('Loss and scheduler defined.')

In [ ]:
"""Checkpoint saving/loading."""

def save_checkpoint(model, optimizer, epoch, step, val_acc, path):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
                'epoch': epoch, 'step': step, 'val_acc': val_acc}, path)
    print(f'  Checkpoint saved: {path} (epoch={epoch}, val_acc={val_acc:.4f})')

def load_checkpoint(model, path, optimizer=None, device='cuda'):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    if optimizer and 'optimizer_state' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state'])
    print(f'  Loaded: {path} (epoch={ckpt.get("epoch", "?")}, val_acc={ckpt.get("val_acc", "?")})')
    return ckpt

print('Checkpoint utils defined.')

In [ ]:
"""Validation: exact-match accuracy on held-out training tasks."""

@torch.no_grad()
def validate(model, val_loader, device='cuda'):
    model.eval()
    total_tasks, solved_tasks, total_cell_correct, total_cells = 0, 0, 0, 0
    for batch in val_loader:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        all_logits = model(
            context_tokens=batch['context_tokens'], context_rows=batch['context_rows'],
            context_cols=batch['context_cols'], context_pairs=batch['context_pairs'],
            context_mask=batch['context_mask'], target_rows=batch['target_rows'],
            target_cols=batch['target_cols'], target_mask=batch['target_mask'],
            test_input_tokens=batch['test_input_tokens'], test_input_rows=batch['test_input_rows'],
            test_input_cols=batch['test_input_cols'], test_input_mask=batch['test_input_mask'])
        preds = all_logits[-1].argmax(dim=-1)
        target = batch['target_tokens']
        mask = batch['target_mask']
        for i in range(preds.shape[0]):
            valid = mask[i]
            pred_cells = preds[i][valid]
            true_cells = target[i][valid]
            correct = (pred_cells == true_cells).all().item()
            solved_tasks += correct
            total_tasks += 1
            total_cell_correct += (pred_cells == true_cells).sum().item()
            total_cells += valid.sum().item()
    model.train()
    return {'exact_match': solved_tasks / max(1, total_tasks),
            'cell_accuracy': total_cell_correct / max(1, total_cells),
            'solved': solved_tasks, 'total': total_tasks}

print('Validation defined.')

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

# Hyperparameters
EPOCHS = 50
BATCH_SIZE = 4
GRAD_ACCUM = 8
LR = 3e-4
WEIGHT_DECAY = 0.01
VALIDATE_EVERY = 5
USE_AMP = True

# Create datasets and dataloaders
train_ds = ARCDataset(train_tasks, augment=True, n_dihedral=8, n_color_perms=2)
val_ds = ARCDataset(val_tasks, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, collate_fn=collate_arc_batch, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, collate_fn=collate_arc_batch, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train dataset: {len(train_ds)} examples ({len(train_tasks)} tasks x {train_ds.augments_per_task} augments)')
print(f'Val dataset: {len(val_ds)} examples')
print(f'Train batches/epoch: {len(train_loader)}')

# Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
total_steps = EPOCHS * len(train_loader) // GRAD_ACCUM
warmup_steps = int(0.05 * total_steps)
scheduler = WSDScheduler(optimizer, warmup_steps, total_steps)
scaler = GradScaler(enabled=USE_AMP)

best_val_acc = 0.0
global_step = 0
train_losses = []
val_metrics_history = []

print(f'\nTraining config:')
print(f'  Device: {device}')
print(f'  Epochs: {EPOCHS}')
print(f'  Effective batch: {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Total steps: {total_steps}, warmup: {warmup_steps}')
print(f'  Parameters: {model.count_parameters():,}')
print(f'  Refinement steps: {model.refine_steps}')
print('-' * 60)

# Try loading existing checkpoint
best_ckpt_path = os.path.join(CHECKPOINT_DIR, 'checkpoint_best.pt')
if os.path.exists(best_ckpt_path):
    ckpt = load_checkpoint(model, best_ckpt_path, optimizer, device)
    best_val_acc = ckpt.get('val_acc', 0.0)
    start_epoch = ckpt.get('epoch', 0) + 1
    print(f'  Resuming from epoch {start_epoch}')
else:
    start_epoch = 1

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_loss = 0.0
    t0 = time.time()

    for batch_idx, batch in enumerate(train_loader):
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_AMP):
            all_logits = model(
                context_tokens=batch['context_tokens'], context_rows=batch['context_rows'],
                context_cols=batch['context_cols'], context_pairs=batch['context_pairs'],
                context_mask=batch['context_mask'], target_rows=batch['target_rows'],
                target_cols=batch['target_cols'], target_mask=batch['target_mask'],
                test_input_tokens=batch['test_input_tokens'], test_input_rows=batch['test_input_rows'],
                test_input_cols=batch['test_input_cols'], test_input_mask=batch['test_input_mask'],
                task_idx=batch['task_idx'])
            loss = deep_supervised_loss(all_logits, batch['target_tokens'], batch['target_mask'])
            loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (batch_idx + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1
        epoch_loss += loss.item() * GRAD_ACCUM

    elapsed = time.time() - t0
    avg_loss = epoch_loss / max(1, len(train_loader))
    lr_now = scheduler.get_lr()[0]
    train_losses.append(avg_loss)
    print(f'Epoch {epoch:3d}/{EPOCHS} | loss={avg_loss:.4f} | lr={lr_now:.2e} | time={elapsed:.1f}s')

    # Validation
    if epoch % VALIDATE_EVERY == 0 or epoch == EPOCHS:
        vm = validate(model, val_loader, device)
        val_metrics_history.append((epoch, vm))
        print(f'  Val: exact_match={vm["exact_match"]:.4f} ({vm["solved"]}/{vm["total"]}) cell_acc={vm["cell_accuracy"]:.4f}')
        if vm['exact_match'] >= best_val_acc:
            best_val_acc = vm['exact_match']
            save_checkpoint(model, optimizer, epoch, global_step, best_val_acc, best_ckpt_path)

# Save final checkpoint
save_checkpoint(model, optimizer, EPOCHS, global_step, best_val_acc,
                os.path.join(CHECKPOINT_DIR, 'checkpoint_final.pt'))
print(f'\nTraining complete. Best val accuracy: {best_val_acc:.4f}')

In [ ]:
# Plot training loss curve
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training Loss')
ax1.grid(True)

if val_metrics_history:
    epochs_v = [e for e, _ in val_metrics_history]
    em = [m['exact_match'] for _, m in val_metrics_history]
    ca = [m['cell_accuracy'] for _, m in val_metrics_history]
    ax2.plot(epochs_v, em, 'o-', label='Exact Match')
    ax2.plot(epochs_v, ca, 's-', label='Cell Accuracy')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Validation Accuracy')
    ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.show()

## Section 4: Evaluation on 400 Held-Out Tasks

In [ ]:
"""Test-Time Training (TTT) and Evaluation."""

def test_time_train(model, task, n_steps=100, lr=1e-4, device='cuda'):
    """
    Fine-tune model copy on demo pairs of a single task at inference time.
    For each demo, treat (input->output) as a training signal without peeking at test output.
    """
    ttt_model = copy.deepcopy(model)
    ttt_model.to(device)
    ttt_model.train()
    optimizer = torch.optim.AdamW(ttt_model.parameters(), lr=lr, weight_decay=0.0)

    for step in range(n_steps):
        total_loss = torch.tensor(0.0, device=device)
        n_demos = len(task.train_pairs)
        for hold_out_idx in range(n_demos):
            held_pair = task.train_pairs[hold_out_idx]
            context_pairs = task.train_pairs[:hold_out_idx] + task.train_pairs[hold_out_idx+1:]
            if len(context_pairs) == 0:
                context_pairs = task.train_pairs
            sub_task = ARCTask(task_id=task.task_id, train_pairs=context_pairs, test_pairs=[held_pair])
            seq = build_task_sequence(sub_task, test_idx=0)
            batch = {k: v.unsqueeze(0).to(device) if isinstance(v, torch.Tensor) else v for k, v in seq.items()}
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                all_logits = ttt_model(
                    context_tokens=batch['context_tokens'], context_rows=batch['context_rows'],
                    context_cols=batch['context_cols'], context_pairs=batch['context_pairs'],
                    context_mask=torch.ones_like(batch['context_tokens'], dtype=torch.bool),
                    target_rows=batch['target_rows'], target_cols=batch['target_cols'],
                    target_mask=torch.ones(1, batch['target_tokens'].shape[1], dtype=torch.bool, device=device),
                    test_input_tokens=batch['test_input_tokens'], test_input_rows=batch['test_input_rows'],
                    test_input_cols=batch['test_input_cols'],
                    test_input_mask=torch.ones_like(batch['test_input_tokens'], dtype=torch.bool))
                logits = all_logits[-1]
                ce = F.cross_entropy(logits.reshape(-1, NUM_COLORS), batch['target_tokens'].reshape(-1))
                total_loss = total_loss + ce / n_demos
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(ttt_model.parameters(), max_norm=1.0)
        optimizer.step()
    ttt_model.eval()
    return ttt_model


@torch.no_grad()
def predict_task(model, task, test_idx=0, device='cuda', T=None):
    """Generate prediction for a single task's test input."""
    model.eval()
    seq = build_task_sequence(task, test_idx=test_idx)
    batch = {k: v.unsqueeze(0).to(device) if isinstance(v, torch.Tensor) else v for k, v in seq.items()}
    all_logits = model(
        context_tokens=batch['context_tokens'], context_rows=batch['context_rows'],
        context_cols=batch['context_cols'], context_pairs=batch['context_pairs'],
        context_mask=torch.ones_like(batch['context_tokens'], dtype=torch.bool),
        target_rows=batch['target_rows'], target_cols=batch['target_cols'],
        target_mask=torch.ones(1, batch['target_tokens'].shape[1], dtype=torch.bool, device=device),
        test_input_tokens=batch['test_input_tokens'], test_input_rows=batch['test_input_rows'],
        test_input_cols=batch['test_input_cols'],
        test_input_mask=torch.ones_like(batch['test_input_tokens'], dtype=torch.bool), T=T)
    preds = all_logits[-1].argmax(dim=-1).squeeze(0).cpu().numpy()
    out_h, out_w = seq['output_h'], seq['output_w']
    return preds[:out_h * out_w].reshape(out_h, out_w).tolist()

print('TTT and prediction functions defined.')

In [ ]:
# ============================================================
# FINAL EVALUATION on 400 evaluation tasks
# ============================================================
# IMPORTANT: We load eval task inputs only.
# Outputs are present in JSON but used ONLY for scoring.

# Load best checkpoint
load_checkpoint(model, best_ckpt_path, device=device)
model.eval()

eval_tasks = load_tasks(EVAL_DIR)
print(f'Loaded {len(eval_tasks)} evaluation tasks.')

USE_TTT = True
TTT_STEPS = 100
TTT_LR = 1e-4

results = []
solved_count = 0

for i, task in enumerate(eval_tasks):
    task_result = {'task_id': task.task_id, 'solved': False, 'attempt1': False, 'attempt2': False}
    for test_idx in range(len(task.test_pairs)):
        true_output = task.test_pairs[test_idx]['output']
        # Attempt 1: base model
        pred1 = predict_task(model, task, test_idx=test_idx, device=device)
        if pred1 == true_output:
            task_result['attempt1'] = True
            task_result['solved'] = True
        # Attempt 2: TTT
        if USE_TTT and not task_result['solved']:
            ttt_model = test_time_train(model, task, n_steps=TTT_STEPS, lr=TTT_LR, device=device)
            pred2 = predict_task(ttt_model, task, test_idx=test_idx, device=device)
            del ttt_model; torch.cuda.empty_cache()
            if pred2 == true_output:
                task_result['attempt2'] = True
                task_result['solved'] = True
    if task_result['solved']:
        solved_count += 1
    results.append(task_result)
    if (i + 1) % 20 == 0:
        print(f'  Evaluated {i+1}/{len(eval_tasks)}, solved: {solved_count}/{i+1} ({solved_count/(i+1)*100:.1f}%)')

accuracy = solved_count / len(eval_tasks)
a1 = sum(1 for r in results if r['attempt1'])
a2 = sum(1 for r in results if r['attempt2'] and not r['attempt1'])

print(f'\n{"="*50}')
print(f'EVALUATION RESULTS')
print(f'{"="*50}')
print(f'Total tasks:     {len(eval_tasks)}')
print(f'Solved:          {solved_count}')
print(f'Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'  Attempt 1:     {a1}')
print(f'  Attempt 2:     {a2} (additional via TTT)')
print(f'{"="*50}')

## Section 5: Ablation Experiments

Each ablation trains a variant and measures exact-match accuracy.

| # | Design Choice | Full Model | Baseline |
|---|---|---|---|
| A | Positional Encoding | 3D RoPE (row, col, pair) | 2D RoPE (row, col only) |
| B | Loss Function | Deep supervision (all T steps) | Final step only |
| C | Per-task Embeddings | 400 learnable vectors | No per-task embeddings |

In [ ]:
# ============================================================
# ABLATION A: 2D RoPE (no pair index encoding)
# ============================================================
# To run: change the pair_index positions to zeros in build_task_sequence,
# then retrain. This tests whether encoding the pair index helps the model
# distinguish between different demo pairs.

# Uncomment and run for ablation:
# ablation_a_model = ARCModel(d_model=384, enc_layers=8, dec_layers=6, n_heads=8,
#     d_ff=1536, n_train_tasks=len(train_tasks), dropout=0.1, refine_steps=5)
# ... (retrain with pair positions zeroed out)
print('Ablation A: 2D RoPE — zero out pair_index in context_pairs')
print('Expected: lower accuracy due to cross-pair confusion')

In [ ]:
# ============================================================
# ABLATION B: Final-step-only loss (no deep supervision)
# ============================================================

def final_step_loss(all_logits, target, mask):
    """CE loss on only the final refinement step."""
    logits = all_logits[-1]
    B, L, C = logits.shape
    flat_logits = logits.reshape(-1, C)
    flat_target = target.reshape(-1)
    flat_mask = mask.reshape(-1).float()
    ce = F.cross_entropy(flat_logits, flat_target, reduction='none')
    return (ce * flat_mask).sum() / flat_mask.sum().clamp(min=1)

# Uncomment and run for ablation:
# ... (retrain with final_step_loss instead of deep_supervised_loss)
print('Ablation B: Final-step-only loss')
print('Expected: significantly lower accuracy — deep supervision is the primary driver')

In [ ]:
# ============================================================
# ABLATION C: No per-task embeddings
# ============================================================

# Uncomment and run for ablation:
# ... (retrain with task_idx=None during training, removing per-task conditioning)
print('Ablation C: No per-task embeddings')
print('Expected: slightly lower accuracy — task embeddings encode task-specific structure')

## Visualization: Sample Predictions

In [ ]:
"""Visualize sample predictions vs ground truth."""
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

ARC_CMAP = mcolors.ListedColormap([
    '#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
    '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'
])
ARC_NORM = mcolors.BoundaryNorm(range(11), 10)

def plot_task_prediction(task, prediction=None):
    n_demos = len(task.train_pairs)
    n_cols = n_demos + 1 + (1 if prediction else 0)
    fig, axes = plt.subplots(2, n_cols, figsize=(3*n_cols, 6))
    if n_cols == 1:
        axes = axes.reshape(2, 1)

    for i, pair in enumerate(task.train_pairs):
        axes[0, i].imshow(pair['input'], cmap=ARC_CMAP, norm=ARC_NORM)
        axes[0, i].set_title(f'Demo {i+1} In')
        axes[1, i].imshow(pair['output'], cmap=ARC_CMAP, norm=ARC_NORM)
        axes[1, i].set_title(f'Demo {i+1} Out')

    axes[0, n_demos].imshow(task.test_pairs[0]['input'], cmap=ARC_CMAP, norm=ARC_NORM)
    axes[0, n_demos].set_title('Test In')
    axes[1, n_demos].imshow(task.test_pairs[0]['output'], cmap=ARC_CMAP, norm=ARC_NORM)
    axes[1, n_demos].set_title('Expected')

    if prediction:
        axes[0, n_demos+1].axis('off')
        axes[1, n_demos+1].imshow(prediction, cmap=ARC_CMAP, norm=ARC_NORM)
        match = prediction == task.test_pairs[0]['output']
        axes[1, n_demos+1].set_title(f'Predicted ({"CORRECT" if match else "wrong"})')

    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'Task {task.task_id}', fontsize=14)
    plt.tight_layout()
    plt.show()

# Show a few sample predictions
model.eval()
for task in val_tasks[:3]:
    pred = predict_task(model, task, device=device)
    plot_task_prediction(task, pred)